# Video Game Sales and Engagement Analysis

## Exploratory Data Analysis (EDA)

This notebook contains the exploratory analysis for the Video Game Sales and Engagement Analysis project.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 2. Load Cleaned Datasets

In [ ]:
games = pd.read_csv('../data/processed/games_cleaned.csv')
sales = pd.read_csv('../data/processed/vgsales_cleaned.csv')

print('Games shape:', games.shape)
print('Sales shape:', sales.shape)

## 3. Dataset Overview

In [ ]:
display(games.head())
display(sales.head())

## 4. Games Dataset Structure

In [ ]:
games.info()

## 5. Sales Dataset Structure

In [ ]:
sales.info()

## 6. Missing Value Check

In [ ]:
print('Missing values in Games dataset:')
display(games.isnull().sum())

print('Missing values in Sales dataset:')
display(sales.isnull().sum())

## 7. Descriptive Statistics

In [ ]:
display(games.describe(include='all').T)
display(sales.describe(include='all').T)

# EDA Questions

The following sections will answer the 30 analytical questions specified in the project requirements.

## Games Analysis — Q1 to Q9

### Q1. What are the top-rated games by user reviews?

In [ ]:
top_rated = games[["Title", "Rating", "Number_of_Reviews"]].dropna(subset=["Rating"])
top_rated = top_rated.sort_values(
    ["Rating", "Number_of_Reviews"],
    ascending=[False, False]
).head(10)

display(top_rated)

**Interpretation:** This table identifies games with the highest recorded user ratings, with number of reviews used as a secondary ordering field.

### Q2. Which developers (Teams) have the highest average ratings?

In [ ]:
team_rating = games.dropna(subset=["Rating"]).groupby("Team").agg(
    Average_Rating=("Rating", "mean"),
    Game_Count=("Title", "count")
).sort_values("Average_Rating", ascending=False)

display(team_rating.head(10))

**Interpretation:** Developer teams are compared using their average recorded user rating. Game count is included to provide context.

### Q3. What are the most common genres in the dataset?

In [ ]:
genre_counts = (
    games["Genres"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
)

display(genre_counts.head(10))

plt.figure(figsize=(10, 6))
genre_counts.head(10).sort_values().plot(kind="barh")
plt.title("Most Common Game Genres")
plt.xlabel("Number of Records")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()

**Interpretation:** Genre frequency shows which categories appear most often in the Games metadata dataset.

### Q4. Which games have the highest backlog compared to wishlist?

In [ ]:
backlog_analysis = games[["Title", "Backlogs", "Wishlist"]].copy()

backlog_analysis["Backlog_to_Wishlist"] = (
    backlog_analysis["Backlogs"] /
    backlog_analysis["Wishlist"].replace(0, np.nan)
)

backlog_analysis = backlog_analysis.dropna(
    subset=["Backlog_to_Wishlist"]
)

display(
    backlog_analysis
    .sort_values("Backlog_to_Wishlist", ascending=False)
    .head(10)
)

**Interpretation:** The ratio compares backlog volume with wishlist volume. A higher ratio indicates relatively more backlog entries than wishlist entries.

### Q5. What is the game release trend across years?

In [ ]:
release_trend = (
    games.dropna(subset=["Release_Year"])
    .groupby("Release_Year")
    .size()
)

display(release_trend.to_frame("Game_Count"))

plt.figure(figsize=(12, 6))
release_trend.plot(kind="line", marker="o")
plt.title("Game Release Trend Across Years")
plt.xlabel("Release Year")
plt.ylabel("Number of Games")
plt.grid(True)
plt.tight_layout()
plt.show()

**Interpretation:** The line chart shows how the number of games represented in the metadata dataset changes across release years.

### Q6. What is the distribution of user ratings?

In [ ]:
rating_data = games["Rating"].dropna()

print("Mean rating:", round(rating_data.mean(), 2))
print("Median rating:", round(rating_data.median(), 2))

plt.figure(figsize=(10, 6))
sns.histplot(rating_data, bins=20, kde=True)
plt.title("Distribution of User Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Games")
plt.tight_layout()
plt.show()

**Interpretation:** The histogram shows the distribution and concentration of recorded user ratings.

### Q7. What are the top 10 most wishlisted games?

In [ ]:
top_wishlisted = (
    games[["Title", "Wishlist"]]
    .sort_values("Wishlist", ascending=False)
    .head(10)
)

display(top_wishlisted)

plt.figure(figsize=(10, 6))
plot_data = top_wishlisted.sort_values("Wishlist")
plt.barh(plot_data["Title"], plot_data["Wishlist"])
plt.title("Top 10 Most Wishlisted Games")
plt.xlabel("Wishlist Count")
plt.tight_layout()
plt.show()

**Interpretation:** These are the ten games with the highest recorded wishlist counts in the Games dataset.

### Q8. What’s the average number of plays per genre?

In [ ]:
plays_genre = games.dropna(
    subset=["Genres", "Plays"]
).copy()

plays_genre["Genre"] = plays_genre["Genres"].str.split(",")
plays_genre = plays_genre.explode("Genre")
plays_genre["Genre"] = plays_genre["Genre"].str.strip()

average_plays = (
    plays_genre.groupby("Genre")["Plays"]
    .mean()
    .sort_values(ascending=False)
)

display(average_plays.to_frame("Average_Plays"))

**Interpretation:** Average plays are calculated after expanding records containing multiple genres so that each listed genre contributes to its genre-level average.

### Q9. Which developer studios are the most productive and impactful?

In [ ]:
studio_analysis = games.groupby("Team").agg(
    Games_Released=("Title", "count"),
    Average_Rating=("Rating", "mean"),
    Total_Reviews=("Number_of_Reviews", "sum"),
    Total_Plays=("Plays", "sum")
).sort_values(
    ["Games_Released", "Average_Rating"],
    ascending=[False, False]
)

display(studio_analysis.head(10))

**Interpretation:** Productivity is represented by the number of games in the dataset. Impact is described using accompanying engagement indicators such as average rating, total reviews, and total plays.

## Q1–Q9 Games Analysis Completed

## Sales Analysis — Q10 to Q20

This section analyzes the cleaned `vgsales` dataset.

The analysis covers regional sales, platform performance, yearly sales trends,
publishers, individual best-selling games, regional-platform relationships,
platform evolution, regional genre preferences, yearly regional changes,
publisher-level averages, and platform-specific best sellers.

**Unit of sales:** million units, as represented in the source dataset.

**Important data limitation:** The dataset becomes sparse in the most recent
years. Therefore, very low sales values in 2017 and 2020 should not be
interpreted as a confirmed industry decline.

### Q10. Which region generates the most game sales?

Regional sales are calculated by summing the four regional sales columns:
`NA_Sales`, `EU_Sales`, `JP_Sales`, and `Other_Sales`.

In [ ]:
regional_sales = {
    "North America": sales["NA_Sales"].sum(),
    "Europe": sales["EU_Sales"].sum(),
    "Japan": sales["JP_Sales"].sum(),
    "Other": sales["Other_Sales"].sum()
}

q10 = (
    pd.Series(regional_sales, name="Sales_Millions")
    .sort_values(ascending=False)
)

display(q10.to_frame())

**Interpretation:** North America records the largest cumulative
sales in the dataset, followed by Europe, Japan, and Other regions.

This describes cumulative sales within this dataset and does not by itself
represent current market share.

### Q11. What are the best-selling platforms?

Platform performance is measured using total `Global_Sales`.
`Game_Count` is included to provide context for the sales totals.

In [ ]:
q11 = (
    sales
    .groupby("Platform")
    .agg(
        Total_Global_Sales=("Global_Sales", "sum"),
        Game_Count=("Name", "count")
    )
    .sort_values("Total_Global_Sales", ascending=False)
)

display(q11.head(15))

**Interpretation:** PS2 has the largest cumulative global sales in
this dataset. Other high-volume platforms include X360, PS3, Wii, DS, and PS.

Platform totals are influenced by both the number of games represented and
their sales performance.

### Q12. What is the trend of game releases and sales over years?

The analysis groups records by release year and calculates both the number of
game records and total global sales.

In [ ]:
q12 = (
    sales
    .dropna(subset=["Year"])
    .groupby("Year")
    .agg(
        Game_Releases=("Name", "count"),
        Global_Sales=("Global_Sales", "sum")
    )
    .reset_index()
    .sort_values("Year")
)

display(q12)

**Interpretation:** Game releases and recorded global sales increase
substantially through the 2000s, with high activity around the late 2000s and
early 2010s.

The dataset has very limited observations for the most recent years. In
particular, 2017 and 2020 contain very few records, so their low sales totals
should be treated as a data-coverage limitation rather than automatically
interpreted as a market decline.

### Q13. Who are the top publishers by sales?

Publishers are ranked by the sum of their recorded `Global_Sales`.

In [ ]:
q13 = (
    sales
    .groupby("Publisher")
    .agg(
        Total_Global_Sales=("Global_Sales", "sum"),
        Game_Count=("Name", "count")
    )
    .sort_values("Total_Global_Sales", ascending=False)
)

display(q13.head(15))

**Interpretation:** Nintendo has the largest cumulative global sales
in the dataset, followed by Electronic Arts and Activision.

The `Game_Count` column provides context because publishers have different
numbers of records in the dataset.

### Q14. Which games are the top 10 best-sellers globally?

The ranking uses the `Global_Sales` value recorded for each game/platform
record in the sales dataset.

In [ ]:
q14 = (
    sales[
        ["Name", "Platform", "Year", "Publisher", "Global_Sales"]
    ]
    .sort_values("Global_Sales", ascending=False)
    .head(10)
)

display(q14)

**Interpretation:** Wii Sports has the largest recorded global sales
value in the dataset, followed by Super Mario Bros. and Mario Kart Wii.

These are sales records as represented in the source dataset.

### Q15. How do regional sales compare for specific platforms?

Regional sales are aggregated by platform to compare North America, Europe,
Japan, and Other regions.

In [ ]:
q15 = (
    sales
    .groupby("Platform")
    .agg(
        North_America=("NA_Sales", "sum"),
        Europe=("EU_Sales", "sum"),
        Japan=("JP_Sales", "sum"),
        Other=("Other_Sales", "sum"),
        Global_Sales=("Global_Sales", "sum")
    )
    .sort_values("Global_Sales", ascending=False)
)

display(q15.head(15))

**Interpretation:** Regional sales patterns differ across platforms.
For example, PS2 has large sales across North America and Europe, while
platforms such as NES, GB, and 3DS show comparatively substantial Japanese
sales within the dataset.

### Q16. How has the market evolved by platform over time?

Sales are grouped by both year and platform. `Game_Count` shows the number of
records represented for each platform-year combination.

In [ ]:
q16 = (
    sales
    .dropna(subset=["Year"])
    .groupby(["Year", "Platform"])
    .agg(
        Global_Sales=("Global_Sales", "sum"),
        Game_Count=("Name", "count")
    )
    .reset_index()
    .sort_values(
        ["Year", "Global_Sales"],
        ascending=[True, False]
    )
)

display(q16)

**Interpretation:** The platform market changes over time as older
platforms decline and newer generations appear. The dataset shows transitions
from early systems such as Atari 2600 and NES through PlayStation, PS2, Xbox,
Wii/DS, and later PS4/Xbox One platforms.

Because the complete platform-year table is large, dashboard visualizations
should be used to make these transitions easier to explore.

### Q17. What are the regional genre preferences?

Genre sales are aggregated separately for each geographic region.

In [ ]:
q17 = (
    sales
    .groupby("Genre")
    .agg(
        North_America=("NA_Sales", "sum"),
        Europe=("EU_Sales", "sum"),
        Japan=("JP_Sales", "sum"),
        Other=("Other_Sales", "sum"),
        Global_Sales=("Global_Sales", "sum")
    )
    .sort_values("Global_Sales", ascending=False)
)

display(q17)

**Interpretation:** Action and Sports have high sales across several
regions. Role-Playing games show a comparatively large Japanese sales
component, while Shooter sales are much more concentrated in North America
and Europe within this dataset.

### Q18. What is the yearly sales change per region?

Year-over-year absolute changes are calculated separately for each region.

In [ ]:
q18 = (
    sales
    .dropna(subset=["Year"])
    .groupby("Year")
    .agg(
        North_America=("NA_Sales", "sum"),
        Europe=("EU_Sales", "sum"),
        Japan=("JP_Sales", "sum"),
        Other=("Other_Sales", "sum"),
        Global_Sales=("Global_Sales", "sum")
    )
    .reset_index()
    .sort_values("Year")
)

q18["NA_Change"] = q18["North_America"].diff()
q18["EU_Change"] = q18["Europe"].diff()
q18["JP_Change"] = q18["Japan"].diff()
q18["Other_Change"] = q18["Other"].diff()

display(q18)

**Interpretation:** Regional sales fluctuate substantially across
years. The strongest increases occur during periods of major platform and
market expansion, while later decreases should be interpreted alongside the
dataset's declining year coverage.

The first year has no previous year, so its change value is naturally `NaN`.

### Q19. What is the average sales per publisher?

Average global sales are calculated as total recorded global sales divided by
the number of sales records for each publisher.

`Game_Count` is included because averages based on very small catalogues can
be unstable.

In [ ]:
q19 = (
    sales
    .groupby("Publisher")
    .agg(
        Average_Global_Sales=("Global_Sales", "mean"),
        Total_Global_Sales=("Global_Sales", "sum"),
        Game_Count=("Name", "count")
    )
    .sort_values("Average_Global_Sales", ascending=False)
)

display(q19.head(15))

**Interpretation:** Publishers with only one or a few records can
have high average sales because their calculation is based on a very small
sample. Therefore, `Average_Global_Sales` should always be considered
alongside `Game_Count` and `Total_Global_Sales`.

### Q20. What are the top 5 best-selling games per platform?

For each platform, records are sorted by global sales and the five highest
sales records are retained.

In [ ]:
q20 = (
    sales
    .sort_values(
        ["Platform", "Global_Sales"],
        ascending=[True, False]
    )
    .groupby("Platform")
    .head(5)
)

display(
    q20[
        ["Platform", "Name", "Year", "Global_Sales"]
    ]
)

**Interpretation:** The top-selling titles differ considerably by
platform. Examples include Wii Sports for Wii, Grand Theft Auto: San Andreas
for PS2, and Pokémon Red/Pokémon Blue for Game Boy.

This question is useful for platform-specific product and franchise analysis.

# Combined Dataset Analysis — Q21 to Q30

This section combines the cleaned game engagement dataset with the cleaned sales dataset.

The analysis uses controlled title-level matching through `Normalized_Title` and
`Normalized_Name` to reduce many-to-many duplication.

**Matched title-level records:** 486

The questions examine relationships between genres, ratings, engagement,
wishlists, platforms and regional sales.


## Combined Dataset Preparation

Before answering Q21–Q30, the two datasets are aggregated at title level.

This is important because a direct merge can duplicate records when multiple
rows represent the same normalized game title.


In [ ]:
# Title-level aggregation

games_agg = (
    games.groupby("Normalized_Title", as_index=False)
    .agg({
        "Title": "first",
        "Rating": "mean",
        "Times Listed": "max",
        "Number of Reviews": "max",
        "Plays": "max",
        "Playing": "max",
        "Backlogs": "max",
        "Wishlist": "max",
        "Genres": "first"
    })
)

sales_agg = (
    sales.groupby("Normalized_Name", as_index=False)
    .agg({
        "Global_Sales": "sum",
        "NA_Sales": "sum",
        "EU_Sales": "sum",
        "JP_Sales": "sum",
        "Other_Sales": "sum"
    })
)

merged = games_agg.merge(
    sales_agg,
    left_on="Normalized_Title",
    right_on="Normalized_Name",
    how="inner"
)

print("Matched title-level records:", len(merged))
merged.head()


## Q21. Which game genres generate the most global sales?

Genres are split where a game contains multiple genres.

A matched game contributes its global sales to each associated genre.


In [ ]:
genre_sales = []

for _, row in merged.iterrows():

    if pd.isna(row["Genres"]):
        continue

    for genre in str(row["Genres"]).split(","):

        genre = genre.strip()

        if genre:
            genre_sales.append({
                "Genre": genre,
                "Global_Sales": row["Global_Sales"]
            })

q21 = pd.DataFrame(genre_sales)

q21_result = (
    q21.groupby("Genre")
    .agg(
        Total_Global_Sales=("Global_Sales", "sum"),
        Game_Count=("Global_Sales", "count")
    )
    .sort_values("Total_Global_Sales", ascending=False)
)

q21_result


**Interpretation:** Adventure records generate the highest total global
sales in the matched dataset, followed by Shooter and Platform.

Because multi-genre games are counted under each associated genre, these totals
should not be interpreted as mutually exclusive market shares.


## Q22. How does user rating affect global sales?

The relationship is examined using rating groups and a Pearson correlation
between rating and global sales.


In [ ]:
rating_data = merged.dropna(
    subset=["Rating", "Global_Sales"]
).copy()

rating_data["Rating_Group"] = pd.cut(
    rating_data["Rating"],
    bins=[0, 1, 2, 3, 4, 5],
    labels=[
        "Below 1",
        "1–1.99",
        "2–2.99",
        "3–3.99",
        "4–4.99"
    ],
    include_lowest=True
)

q22_result = (
    rating_data.groupby("Rating_Group", observed=False)
    .agg(
        Game_Count=("Global_Sales", "count"),
        Average_Global_Sales=("Global_Sales", "mean"),
        Total_Global_Sales=("Global_Sales", "sum"),
        Average_Rating=("Rating", "mean")
    )
)

rating_corr = rating_data[
    ["Rating", "Global_Sales"]
].corr().iloc[0, 1]

print(q22_result)
print("\nRating vs Global Sales correlation:", round(rating_corr, 4))


**Interpretation:** The rating-to-sales correlation in this matched
sample is approximately **0.0121**. This indicates that the two variables have
almost no linear relationship in this dataset.

Correlation should not be interpreted as causation.


## Q23. Which platforms have the most games with high ratings?

A high rating is defined according to the project question as a rating **above 4**.


In [ ]:
q23_data = sales.merge(
    games_agg[
        ["Normalized_Title", "Rating", "Title"]
    ],
    left_on="Normalized_Name",
    right_on="Normalized_Title",
    how="inner"
)

high_rated = q23_data[
    q23_data["Rating"] > 4
]

q23_result = (
    high_rated.groupby("Platform")
    .agg(
        High_Rated_Games=("Normalized_Name", "nunique"),
        Average_Rating=("Rating", "mean")
    )
    .sort_values("High_Rated_Games", ascending=False)
)

q23_result


**Interpretation:** PS2 has the largest number of matched high-rated
game records in this analysis, followed by PC and PS3.

The result is a count of high-rated matched games, not a platform quality score.


## Q24. What’s the trend of releases and sales over time?

Release counts come from `games.csv`, while sales totals come from `vgsales.csv`.

The two datasets have different year coverage, so missing sales values are
retained as missing rather than being treated as zero.


In [ ]:
games_year = (
    games.dropna(subset=["Release_Year"])
    .groupby("Release_Year")
    .agg(
        Game_Releases=("Normalized_Title", "nunique")
    )
)

sales_year = (
    sales.dropna(subset=["Year"])
    .groupby("Year")
    .agg(
        Global_Sales=("Global_Sales", "sum")
    )
)

games_year.index = games_year.index.astype(int)
sales_year.index = sales_year.index.astype(int)

q24_result = (
    games_year
    .join(sales_year, how="outer")
    .sort_index()
)

q24_result


**Interpretation:** The combined yearly view shows increasing release
activity through the 2000s and substantial sales totals during the same period.

Late-year values require caution because the two source datasets have incomplete
and different temporal coverage. Missing sales values do not represent zero sales.


## Q25. Do highly wishlisted games lead to more sales?

The relationship between wishlist counts and global sales is examined using
correlation and wishlist quartiles.


In [ ]:
q25_data = merged.dropna(
    subset=["Wishlist", "Global_Sales"]
).copy()

q25_corr = q25_data[
    ["Wishlist", "Global_Sales"]
].corr().iloc[0, 1]

q25_data["Wishlist_Group"] = pd.qcut(
    q25_data["Wishlist"],
    q=4,
    duplicates="drop"
)

q25_result = (
    q25_data.groupby("Wishlist_Group", observed=False)
    .agg(
        Game_Count=("Global_Sales", "count"),
        Average_Wishlist=("Wishlist", "mean"),
        Average_Global_Sales=("Global_Sales", "mean"),
        Total_Global_Sales=("Global_Sales", "sum")
    )
)

print("Wishlist vs Global Sales correlation:",
      round(q25_corr, 4))

q25_result


**Interpretation:** The wishlist-to-global-sales correlation is
approximately **-0.0691** in the matched sample.

The relationship is therefore weak in this dataset, and higher wishlist counts
do not correspond to a simple increasing sales pattern.

This is an association analysis, not evidence that wishlisting causes lower or
higher sales.


## Q26. Which genres have the highest engagement but lowest sales?

For this project, an engagement measure is constructed as:

**Plays + Playing + Backlogs + Wishlist**

Genres are then compared using their average engagement and average global sales.


In [ ]:
genre_engagement = []

for _, row in merged.iterrows():

    if pd.isna(row["Genres"]):
        continue

    engagement = (
        row["Plays"]
        + row["Playing"]
        + row["Backlogs"]
        + row["Wishlist"]
    )

    for genre in str(row["Genres"]).split(","):

        genre = genre.strip()

        if genre:
            genre_engagement.append({
                "Genre": genre,
                "Engagement": engagement,
                "Global_Sales": row["Global_Sales"]
            })

q26 = pd.DataFrame(genre_engagement)

q26_result = (
    q26.groupby("Genre")
    .agg(
        Average_Engagement=("Engagement", "mean"),
        Average_Global_Sales=("Global_Sales", "mean"),
        Total_Global_Sales=("Global_Sales", "sum"),
        Game_Count=("Global_Sales", "count")
    )
)

q26_result["Engagement_Rank"] = (
    q26_result["Average_Engagement"].rank(pct=True)
)

q26_result["Sales_Rank"] = (
    q26_result["Average_Global_Sales"].rank(pct=True)
)

q26_result["Engagement_Sales_Gap"] = (
    q26_result["Engagement_Rank"]
    - q26_result["Sales_Rank"]
)

q26_result.sort_values(
    "Engagement_Sales_Gap",
    ascending=False
)


**Interpretation:** Genres such as Indie, Turn Based Strategy,
Visual Novel and Point-and-Click show relatively high engagement compared with
their average sales in this matched dataset.

The engagement measure is a project-defined composite metric and should be
interpreted as a comparative indicator rather than an official industry metric.


## Q27. Do highly listed games (wishlist/backlogs) correlate with better ratings?

A combined listing-interest measure is created from:

**Times Listed + Backlogs + Wishlist**

The relationship with user rating is then measured.


In [ ]:
q27_data = merged.dropna(
    subset=[
        "Times Listed",
        "Backlogs",
        "Wishlist",
        "Rating"
    ]
).copy()

q27_data["Total_Listed_Interest"] = (
    q27_data["Times Listed"]
    + q27_data["Backlogs"]
    + q27_data["Wishlist"]
)

q27_corr = q27_data[
    ["Total_Listed_Interest", "Rating"]
].corr().iloc[0, 1]

q27_data["Listing_Group"] = pd.qcut(
    q27_data["Total_Listed_Interest"],
    q=4,
    duplicates="drop"
)

q27_result = (
    q27_data.groupby("Listing_Group", observed=False)
    .agg(
        Game_Count=("Rating", "count"),
        Average_Listed_Interest=(
            "Total_Listed_Interest",
            "mean"
        ),
        Average_Rating=("Rating", "mean")
    )
)

print(
    "Total listed interest vs Rating correlation:",
    round(q27_corr, 4)
)

q27_result


**Interpretation:** Total listed interest and rating have a
correlation of approximately **0.5082** in the matched sample.

The quartile analysis also shows increasing average ratings across the four
listing-interest groups.

This demonstrates association within the dataset and does not establish a
causal relationship.


## Q28. How does user engagement differ across genres?

Average engagement is compared across genres using the same project-defined
engagement measure from Q26.


In [ ]:
q28_result = (
    q26.groupby("Genre")
    .agg(
        Average_Engagement=("Engagement", "mean"),
        Average_Global_Sales=("Global_Sales", "mean"),
        Game_Count=("Global_Sales", "count")
    )
    .sort_values(
        "Average_Engagement",
        ascending=False
    )
)

q28_result


**Interpretation:** Shooter has the highest average engagement in
the matched dataset, followed by Adventure, Indie and Turn Based Strategy.

Genre counts differ substantially, so average engagement should be considered
alongside `Game_Count`.


## Q29. What are the top-performing combinations of Genre + Platform?

The sales dataset is grouped by both genre and platform to identify combinations
with the highest cumulative global sales.


In [ ]:
q29_result = (
    sales.groupby(
        ["Genre", "Platform"]
    )
    .agg(
        Total_Global_Sales=("Global_Sales", "sum"),
        Game_Count=("Global_Sales", "count"),
        Average_Global_Sales=("Global_Sales", "mean")
    )
    .sort_values(
        "Total_Global_Sales",
        ascending=False
    )
)

q29_result.head(30)


**Interpretation:** Among the combinations in the dataset, Action +
PS3 has the largest cumulative global sales, followed by Sports + Wii and
Shooter + X360.

Cumulative sales are affected by the number of records in each
genre-platform combination, so `Game_Count` should be considered alongside
total sales.


## Q30. What does a regional sales heatmap by genre reveal?

Regional sales are aggregated by genre across North America, Europe, Japan and
Other regions.

The resulting table can directly support a heatmap in the visualization stage.


In [ ]:
q30_result = (
    sales.groupby("Genre")
    .agg(
        NA_Sales=("NA_Sales", "sum"),
        EU_Sales=("EU_Sales", "sum"),
        JP_Sales=("JP_Sales", "sum"),
        Other_Sales=("Other_Sales", "sum"),
        Global_Sales=("Global_Sales", "sum")
    )
    .sort_values(
        "Global_Sales",
        ascending=False
    )
)

q30_result


**Interpretation:** North America has the largest sales totals for
many genres, while Japan has a comparatively strong contribution for
Role-Playing and Fighting games.

For example, Role-Playing records show **352.31M** in Japan compared with
**327.28M** in North America.

The regional heatmap should therefore be used to reveal differences in genre
sales patterns rather than treating all regions as having the same preferences.


## Q21–Q30 Summary

The combined analysis examined:

- Genre-level global sales
- Rating and sales relationships
- High-rated games by platform
- Release and sales trends over time
- Wishlist and sales relationships
- Engagement versus sales
- Listing interest and ratings
- Engagement differences across genres
- Genre + platform sales combinations
- Regional sales patterns by genre

### Methodological Notes

1. The merged analysis uses controlled title matching.
2. Multi-genre games can contribute to more than one genre.
3. Engagement is defined for this project as Plays + Playing + Backlogs + Wishlist.
4. Listing interest is defined as Times Listed + Backlogs + Wishlist.
5. Correlation measures association, not causation.
6. Different source datasets have different temporal coverage.
7. Missing yearly sales values are not treated as zero.


# Conclusion

Key findings, patterns, limitations, and analytical observations will be summarized after completing Q1–Q30.